# Scam Call Shield — voice classifier training on AMD GPU (ROCm)

Trains our HUMAN-vs-AI voice CNN on an **AMD Instinct GPU** via ROCm PyTorch.
This is the same code as `voice_classifier/train.py` — ROCm PyTorch exposes the
GPU through the standard `torch.cuda` API, so no code changes are needed.

**Run order:** 1) setup → 2) upload/clone repo → 3) prepare data → 4) train → 5) verify checkpoint in `voice_trends/`.

In [ ]:
# 1. Environment check — should print an AMD GPU name (e.g. 'AMD Instinct MI300X')
import torch
print('torch      :', torch.__version__)
print('gpu ok     :', torch.cuda.is_available())
print('device     :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('is ROCm    :', torch.version.hip is not None, '| hip:', torch.version.hip)

In [ ]:
# If torch is missing or CPU-only on this AMD machine, install ROCm wheels:
# %pip install torch torchaudio --index-url https://download.pytorch.org/whl/rocm6.2
%pip install -q soundfile librosa edge-tts tqdm scikit-learn

In [ ]:
# 2. Get the code (skip if you uploaded the repo manually)
# !git clone https://github.com/<your-user>/scam-call-shield.git
# %cd scam-call-shield
import os
print('working dir:', os.getcwd())
assert os.path.exists('voice_classifier/train.py'), 'run this notebook from the repo root'

In [ ]:
# 3. Build the dataset: LibriSpeech (human) + edge-tts neural voices (AI)
#    ~15-30 min depending on network. Idempotent — safe to re-run.
!python -m voice_classifier.prepare_data --max-human 1200 --max-ai 1200

In [ ]:
# 4. Train on the AMD GPU. Checkpoint + training log land in voice_trends/
!python -m voice_classifier.train --epochs 12 --batch-size 64 --out voice_trends

In [ ]:
# 5. Verify: the log records the GPU it trained on — keep this as AMD-usage evidence
import json
log = json.load(open('voice_trends/training_log.json'))
print('trained on :', log['gpu'])
print('torch      :', log['torch'])
print('best val acc:', log['best_val_acc'])
for h in log['history']:
    print(h)

In [ ]:
# 6. Quick sanity inference on one human + one AI sample
from voice_classifier.infer import VoiceAuthenticity
from pathlib import Path
va = VoiceAuthenticity()
h = next(Path('data/human').glob('*.wav')); a = next(Path('data/ai').glob('*.wav'))
print('human clip -> ai_prob =', round(va.score_file(str(h))[0], 3))
print('ai clip    -> ai_prob =', round(va.score_file(str(a))[0], 3))